# 10 Consensus, agreement and when they mislead

Several voters that agree feel safer than one. This notebook tests that feeling on the Meridian test messages in two steps.
Part 1 asks how often voters agree, and whether agreeing means being right. Part 2 turns the answer into a deployable pattern, the **consensus gate**: act when the voters agree, send the message on when they do not.

The voters are the three encoder heads from notebook 01 (`bge-small`, `modernbert`, `lfm2.5-enc`) and the recorded slow-tier answers from notebook 05. No model is called here: every prediction comes from cached embeddings and a recorded file, so the notebook runs in seconds.

What the numbers cover: 200 test messages, 40 scenarios of 5 variants. The three encoders share one training set, and the slow-tier answers were written by a language model that also helped write the messages. Both facts limit how independent the voters are, and Part 1 measures how much.

In [1]:
import json
import numpy as np
from collections import Counter
from pathlib import Path
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
import meridian as m

RESULTS = Path("data/results")
r01 = json.loads((RESULTS / "01_encoders_and_baseline.json").read_text())
r03 = json.loads((RESULTS / "03_calibration_and_conformal.json").read_text())
tr, te = m.load("train"), m.load("test")
_, ytr = m.texts_labels(tr); _, yte = m.texts_labels(te)

votes = {}
for name in ("bge-small", "modernbert", "lfm2.5-enc"):
    E = m.cached_embeddings(name)
    votes[name] = LogisticRegression(C=r01[name]["C"], max_iter=3000).fit(E["train"], ytr).predict(E["test"])
idmap = json.loads(Path("data/llm/test_id_map.json").read_text())
ans = {idmap[j["id"]]: j["route"] for j in map(json.loads, Path("data/llm/test_haiku_answers.jsonl").read_text().splitlines())}
votes["slow tier"] = np.array([m.ROUTE_ID[ans[r["id"]]] for r in te])

for name, v in votes.items():
    ok = (v == yte); lo, hi = m.wilson(int(ok.sum()), len(ok))
    print(f"{name:11s} accuracy {ok.mean():.3f}  Wilson [{lo:.3f}, {hi:.3f}]  wrong on {int((~ok).sum())}")

bge-small   accuracy 0.970  Wilson [0.936, 0.986]  wrong on 6
modernbert  accuracy 0.925  Wilson [0.880, 0.954]  wrong on 15
lfm2.5-enc  accuracy 0.925  Wilson [0.880, 0.954]  wrong on 15
slow tier   accuracy 0.950  Wilson [0.910, 0.973]  wrong on 10


**Goal.** Put the four voters side by side before combining them. A voting rule can only help if the voters make different mistakes, so the first thing to know is how many mistakes each makes and how well they cover for each other.

**Reading the output.** The strongest voter is `bge-small` (97.0%, 6 wrong); the other two encoders are 92.5% (15 wrong each) and the recorded slow tier is 95.0% (10 wrong).

## Part 1. Agreeing is not the same as being right

For every pair of voters we count how often they give the same answer, how often both are wrong, and how often both are wrong with the same label. The last count is the dangerous one: if two voters are wrong together and agree, a vote between them cannot tell.

**What the next cell does.** For each of the six pairs it prints the share of messages with identical answers, the number of messages both get wrong, and how many of those they get wrong with the same label. It also prints the correlation of the two error patterns (1 means they fail on the same messages, 0 means unrelated, negative means they cover for each other).

In [2]:
names = list(votes)
wrong = {n: votes[n] != yte for n in names}
phis = []
for i, a in enumerate(names):
    for b in names[i + 1:]:
        both = wrong[a] & wrong[b]; same = both & (votes[a] == votes[b])
        phi = float(np.corrcoef(wrong[a], wrong[b])[0, 1]); phis.append(phi)
        print(f"{a:10s} + {b:10s}  same answer {np.mean(votes[a] == votes[b]):.3f}  both wrong {int(both.sum()):2d}  same wrong label {int(same.sum()):2d}  error correlation {phi:+.2f}")
rho = float(np.mean(phis)); n = len(names)
print(f"\nmean error correlation {rho:+.2f}; {n} voters behave like about {n / (1 + (n - 1) * rho):.1f} independent ones")

bge-small  + modernbert  same answer 0.920  both wrong  3  same wrong label  2  error correlation +0.28
bge-small  + lfm2.5-enc  same answer 0.935  both wrong  4  same wrong label  4  error correlation +0.40
bge-small  + slow tier   same answer 0.920  both wrong  0  same wrong label  0  error correlation -0.04
modernbert + lfm2.5-enc  same answer 0.915  both wrong  7  same wrong label  6  error correlation +0.42
modernbert + slow tier   same answer 0.895  both wrong  2  same wrong label  2  error correlation +0.11
lfm2.5-enc + slow tier   same answer 0.925  both wrong  5  same wrong label  5  error correlation +0.37

mean error correlation +0.26; 4 voters behave like about 2.3 independent ones


**What we got.** The three encoders fail together: `modernbert` and `lfm2.5-enc` are both wrong on 7 messages, and on 6 of them they give the same wrong label. The slow tier fails differently: `bge-small` and the slow tier are never wrong on the same message. So a second encoder adds little (same training data, similar mistakes), and a voter of a different kind adds more. The rough rule `n / (1 + (n - 1) x correlation)` says the four voters carry the information of about 2.3 independent ones, close to the 2.0 to 2.5 that Kohli reports for nine LLM judges from seven model families (arXiv:2605.29800, https://arxiv.org/abs/2605.29800); Kim et al. find that two LLMs that both err give the same wrong answer 60% of the time on one leaderboard (arXiv:2506.07962, https://arxiv.org/abs/2506.07962). The formula assumes every pair is equally correlated, so read our 2.3 as a rough guide.

## Part 2. The consensus gate

The pattern is a rule in plain code, not a model. Collect one vote per voter, count the most common answer, and decide by how many voters back it:

- all voters agree: act;
- a strict majority agrees: act, and log the dissent;
- no majority: send the message on, to the slow tier or a person.

A vote can only select a route from the closed list, so the pattern keeps the guarantee the encoders already give: the output is never outside the route list.

**What the next cell does.** `consensus` takes the voters' answers for one message and returns a typed result: the winning route and how many voters back it. `run` applies a rule (the number of votes needed to act) to all 200 test messages and prints what share would act, how many of those are wrong, and how many go on. The first rule needs all 4 voters, the second 3 of 4. The last two lines use only the three cheap encoders, since each extra LLM voter is an extra call per message.

In [3]:
def consensus(answers):
    top, n = Counter(answers).most_common(1)[0]
    return top, n

M = np.stack([votes[n] for n in names]).T            # one row per message, one column per voter
result = [consensus(row) for row in M]
top, cnt = np.array([r[0] for r in result]), np.array([r[1] for r in result])

def run(label, top, cnt, need):
    act = cnt >= need; k, nact = int((top[act] == yte[act]).sum()), int(act.sum())
    lo, hi = m.wilson(k, nact)
    print(f"{label:34s} acts on {nact:3d} of 200 ({nact / 200:.1%}), wrong on {nact - k}, accuracy when it acts {k / nact:.3f} [Wilson {lo:.3f}, {hi:.3f}]; {200 - nact} go on")
    return dict(act=nact, wrong=nact - k, wilson=[lo, hi])

out = {}
out["four_of_four"] = run("4 voters, all must agree", top, cnt, 4)
out["three_of_four"] = run("4 voters, 3 of 4 must agree", top, cnt, 3)
enc = np.stack([votes[n] for n in names[:3]]).T
etop, ecnt = np.array([consensus(r)[0] for r in enc]), np.array([consensus(r)[1] for r in enc])
out["encoders_unanimous"] = run("3 encoders only, all must agree", etop, ecnt, 3)

4 voters, all must agree           acts on 171 of 200 (85.5%), wrong on 0, accuracy when it acts 1.000 [Wilson 0.978, 1.000]; 29 go on
4 voters, 3 of 4 must agree        acts on 191 of 200 (95.5%), wrong on 4, accuracy when it acts 0.979 [Wilson 0.947, 0.992]; 9 go on
3 encoders only, all must agree    acts on 178 of 200 (89.0%), wrong on 2, accuracy when it acts 0.989 [Wilson 0.960, 0.997]; 22 go on


**Goal.** Find out whether voting beats the single best encoder, and what it costs. The comparison points are `bge-small` alone (97.0% on all 200) and the conformal gate from notebook 03 (acts on 91.5%, 98.4% right when it acts).

**What we got.** Requiring all four voters to agree acts on 171 messages and is right on all of them; 29 messages go on. Requiring 3 of 4 acts on 191 and is wrong on 4. The cheap version, three encoders that must all agree, acts on 178 and is wrong on 2, with 22 going on. When they act, the two all-agree rules are at least as accurate as the conformal gate (98.4% on 183 messages); the 3-of-4 rule is slightly below it at 97.9%, a difference well inside the intervals. They act on 85.5% to 95.5% of messages, against 91.5% for the gate. Two costs come with that. The four-voter rules call the slow tier on every message, where the conformal cascade calls it on 8.5%. And the accuracy comes with wide intervals: 0 wrong out of 171 still leaves a Wilson lower end of 97.8%.

## Where unanimity fails

A unanimous vote of the three encoders is not a guarantee. The next cell lists the messages where all three encoders agree and are wrong together.

In [4]:
un = (enc == enc[:, [0]]).all(1)
bad = np.where(un & (enc[:, 0] != yte))[0]
print(f"all three encoders agree on {int(un.sum())} of 200; they are wrong together on {len(bad)}")
for i in bad:
    print(f"  true {m.ROUTES[yte[i]]:20s} voted {m.ROUTES[enc[i, 0]]:20s} slow tier said {m.ROUTES[votes['slow tier'][i]]}")
split = ~un
print(f"\nwhere the encoders disagree ({int(split.sum())} messages): best encoder right on {np.mean(votes['bge-small'][split] == yte[split]):.2f}, slow tier right on {np.mean(votes['slow tier'][split] == yte[split]):.2f}")
out["unanimous_but_wrong"] = int(len(bad)); out["rho"] = rho; out["effective_voters"] = float(n / (1 + (n - 1) * rho))
(RESULTS / "10_consensus_and_agreement.json").write_text(json.dumps(out, indent=2, default=float))

all three encoders agree on 178 of 200; they are wrong together on 2
  true out_of_scope         voted rebooking_request    slow tier said out_of_scope
  true refund_claim         voted payment_dispute      slow tier said refund_claim

where the encoders disagree (22 messages): best encoder right on 0.82, slow tier right on 0.77


481

**What we got.** The encoders agree on 178 messages and are wrong together on 2 of them. For those 2, the slow tier gave the right answer, so a unanimous encoder vote that is checked by a differently built voter would have caught them. Where the encoders disagree (22 messages), the best encoder is right on 82% of them, which is why a split vote is a good signal to send a message on.

**What this shows for the pattern.** Agreement is a warning sign when the voters are alike and a useful signal when they are not. Use voters of different kinds (an encoder and an LLM, or encoders trained on different data), measure how often they fail together on a labeled set like this one, and treat the effective number of voters, not the headcount, as what you bought. Disagreement is cheap to compute on live traffic and needs no labels, so it also works as an alarm: if the share of split votes climbs from 11%, something changed.